In [ ]:
# conda activate anndata

import sys
import pickle
import numpy as np
import pandas as pd

sys.path.append("code")

from process_gtf import *
from empirical_corr_pvals import *

In [ ]:
data_source = "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed"
psi = pd.read_csv(f"data/GTEx_frontal_cortex_SE_PSI.csv", index_col=0)
eigengene_df = pd.read_csv("data/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_ct_eigengenes.csv", index_col=0)

In [ ]:
# Make sure order of samples matches
eigengene_df.index = eigengene_df.index.str.replace(".", "-") 
common = psi.columns.intersection(eigengene_df.index)
psi = psi[common]
eigengene_df = eigengene_df.loc[common]

In [ ]:
# psi_corr_df.to_csv(f"data/{data_source}_ct_eigengene_PSI_corr.csv")

##  Correlate PSI and eigengenes and calc empirical p-values

In [ ]:
# --- correlations + permutation p-values + FDR + CIs ---
(psi_corr_df, psi_pval_df, psi_fdr_df, psi_ci_lower_df, psi_ci_upper_df,
 perm_corr_results, psi_rank_centered, psi_rank_norm, ct_ranks_dict) = spearman_permutation_test(
    psi,
    eigengene_df,
    n_perms=10000,
    ci=0.95
)

Done: CGE Class
Done: All GABAergic
Done: Micro/PVM
Done: Oligo
Done: Astro
Done: Endo
Done: All Neuronal
Done: Deep layer glutamatergic
Done: Upper layer glutamatergic
Done: OPC



In [ ]:

steiger_results = compare_all_ct_pairs(
    psi_corr_df,
    perm_corr_results,
    eigengene_df.columns.tolist()
)

In [ ]:
print("find_specific_SEs_per_ct_basic:")

ctype_specific_SEs_basic = find_specific_SEs_per_ct_basic(
    psi_fdr_df,
    psi_corr_df,
    fdr_thresh=0.05
)
ctype_specific_SEs_strict = {}

print("")
print("find_specific_SEs_per_ct:")

# correlation difference test 
ct_hierarchy = {
    # broader  cell classes should not be comapred against their subtypes (subtypes listed as children);
    'All Neuronal':              ['All GABAergic', 'CGE Class','Upper layer glutamatergic', 'Deep layer glutamatergic'],
    'All GABAergic':             ['CGE Class'],
    # subtypes must be greater than everything
    'CGE Class':                 [],
    'Upper layer glutamatergic': [],
    'Deep layer glutamatergic':  [],
    'Oligo':                     [],
    'OPC':                       [],
    'Astro':                     [],
    'Micro/PVM':                 [],
    'VLMC':                      [],
    'Endo':                      [],
    'Peri':                      [],
}

for ascending in (True, False):
    print("")
    print(ascending)
    ctype_specific_SEs_strict[str(ascending)] = find_specific_SEs_per_ct(
        steiger_results,
        psi_fdr_df,
        psi_corr_df,
        ct_hierarchy,
        fdr_thresh=0.05,
        ascending=ascending
    )

find_specific_SEs_per_ct_basic:
CGE Class: 5804 significant SEs
All GABAergic: 7557 significant SEs
Micro/PVM: 92 significant SEs
Oligo: 609 significant SEs
Astro: 6241 significant SEs
Endo: 2883 significant SEs
All Neuronal: 8582 significant SEs
Deep layer glutamatergic: 5586 significant SEs
Upper layer glutamatergic: 6350 significant SEs
OPC: 417 significant SEs

find_specific_SEs_per_ct:

True
CGE Class: 0 specific SEs (5804 significant total)
All GABAergic: 4 specific SEs (7557 significant total)
Micro/PVM: 7 specific SEs (92 significant total)
Oligo: 17 specific SEs (609 significant total)
Astro: 1006 specific SEs (6241 significant total)
Endo: 3 specific SEs (2883 significant total)
All Neuronal: 1597 specific SEs (8582 significant total)
Deep layer glutamatergic: 2 specific SEs (5586 significant total)
Upper layer glutamatergic: 3 specific SEs (6350 significant total)
OPC: 2 specific SEs (417 significant total)

False
CGE Class: 0 specific SEs (5804 significant total)
All GABAer

In [21]:
# --- combine results ---
combined_results = combine_results(
    ctype_specific_SEs_basic,
    ctype_specific_SEs_strict,
    psi_corr_df,
    psi_fdr_df.drop,
    steiger_results,
    ct_hierarchy,
    SE_coords_df
)

NameError: name 'SE_coords_df' is not defined

In [18]:
ctype_specific_SEs_strict['True']

{'CGE Class': Empty DataFrame
 Columns: [CGE Class, All GABAergic, Micro/PVM, Oligo, Astro, Endo, All Neuronal, Deep layer glutamatergic, Upper layer glutamatergic, OPC]
 Index: [],
 'All GABAergic':                                  CGE Class  All GABAergic  Micro/PVM  \
 ENSG00000132589_ProteinCoding_6  -0.436464      -0.488761   0.017424   
 ENSG00000004866_other_4          -0.359962      -0.428334  -0.092440   
 ENSG00000123349_ProteinCoding_1  -0.394594      -0.422975  -0.051077   
 ENSG00000133424_ProteinCoding_5  -0.306511      -0.349730  -0.076452   
 
                                     Oligo     Astro      Endo  All Neuronal  \
 ENSG00000132589_ProteinCoding_6  0.033372  0.475845  0.359456     -0.354347   
 ENSG00000004866_other_4          0.026703  0.328502  0.190282     -0.272827   
 ENSG00000123349_ProteinCoding_1  0.096740  0.296899  0.188454     -0.307101   
 ENSG00000133424_ProteinCoding_5  0.048451  0.165921  0.082243     -0.199722   
 
                                

In [ ]:
# # Format and save results per cell type

# column_order = ['Gene', 'is_specific', 'specific_direction', 
#                 'chr', 'strand', 'SE_start', 'SE_end', 'SE_len',
#                 'r', 'fdr', 
#                 'CGE Class', 'All GABAergic', 'All Neuronal',
#                 'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
#                 'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
#                 ]

# for ctype, df in combined_results.items():
#     print(ctype)
#     # Remove SEs with NaN correlations
#     df = df[~np.isnan(df['Oligo'])]
#     # Add strand information from GTF
#     df = df.join(
#         gtf_parsed[['gene_name', 'strand']].set_index('gene_name'),
#         on='Gene',
#         how='left'
#     )
#     rest_columns = df.columns[df.columns.str.contains("diff")].tolist() 
#     df[column_order + rest_columns].to_csv(f"data/ctype_SEs/{_safe(ctype)}_SEs.csv")

## Annotate and save results per cell type

In [ ]:
     all_sig['SE_start'] = SE_coords_df.loc[all_sig.index, 'SE_start']
        all_sig['SE_end'] = SE_coords_df.loc[all_sig.index, 'SE_end']
        all_sig['SE_len'] = all_sig['SE_end'] - all_sig['SE_start'] + 1
        all_sig['chr'] = SE_coords_df.loc[all_sig.index, 'chr']

In [ ]:
                # SE coordinates
                start_coord = int(SE_coords_df.loc[idx]['SE_start'])
                end_coord   = int(SE_coords_df.loc[idx]['SE_end'])
                SE_len    = end_coord - start_coord + 1

In [ ]:
exclude = ""
gene_name = "gene_id"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v50.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)

In [ ]:
with(open("data/gencode.v50.annotation_gtf_parsed.pkl", "wb"), pickle.dump(gtf))

In [ ]:
# # Get PSI data ready to merge on gene IDs
# psi['gene_id'] = psi.index.str.split("_").str[0]
# psi['SE_id'] = psi.index.values

# psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
# psi_anno = psi_anno.set_index("SE_id").rename_axis(None)
# psi_anno = psi_anno.drop(columns=["gene_id"])

In [ ]:
# Get PSI data ready to merge on gene IDs
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['SE_id'] = psi.index.values

psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("SE_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [ ]:
def safe_SE_coords(g):
    i1 = g.loc[g.index.str.contains("I1$"), "intron_last_base"].values
    i2 = g.loc[g.index.str.contains("I2$"), "intron_first_base"].values
    if len(i1) == 0 or len(i2) == 0:
        return pd.Series({"chr": None, "SE_start": None, "SE_end": None})
    return pd.Series({
        "chr": g["chr"].iloc[0],
        "SE_start": i1[0] + 1,
        "SE_end": i2[0] - 1
    })


SE_coords_df = intron_coords_df.groupby("SE").apply(safe_SE_coords)
SE_coords_df = SE_coords_df.dropna()  # drop SEs with missing coords

In [ ]:
# Get coordinates for each SE
intron_coords_df = intron_table['intron'].str.split(r"[:\-]", expand=True).iloc[:, :4]
intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base", "strand"]
intron_coords_df.index = intron_table.index
intron_coords_df['SE'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
intron_coords_df = intron_coords_df[intron_coords_df['SE'].isin(psi_anno.index)] # Subset to SEs in PSI data
intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)


In [ ]:
psi_fdr_df.to_csv("data/ct_eigengene_vs_PSI_correlation_FDRs.csv")
psi_corr_df.to_csv("data/SE_ct_eigengene_vs_PSI_correlations.csv")

In [ ]:
# Save significant ctype SEs

column_order = ['Gene', 'is_specific', 'specific_direction', 
                'chr', 'strand', 'SE_start', 'SE_end', 'SE_len',
                'r', 'fdr', 
                'CGE Class', 'All GABAergic', 'All Neuronal',
                'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
                'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
                ]

for ctype, df in combined_results.items():
    print(ctype)
    # Remove SEs with NaN correlations
    df = df[~np.isnan(df['Oligo'])]
    df = df.join(
        gtf_parsed[['gene_name', 'strand']].set_index('gene_name'),
        on='Gene',
        how='left'
    )
    rest_columns = df.columns[df.columns.str.contains("diff")].tolist() 
    df[column_order + rest_columns].to_csv(f"data/ctype_SEs/{_safe(ctype)}_SEs.csv")